<img src='https://github.com/destination-earth/DestinE-DataLake-Lab/blob/main/img/DestinE-banner.jpg?raw=true' align='right' width='100%'/>


<font color="#138D75">**EUMETSAT DestinE User Engagement Service**</font> <br>
**Copyright:** 2026 European Union <br>
**License:** MIT <br>
**Authors:** EUMETSAT DestinE User Engagement Service: Ben Loveday (EUMETSAT/Innoflair UG), Joana Brito (EUMETSAT/Innoflair UG), Madalina Ungur (EUMETSAT/Solenix)

<div class="alert alert-block alert-success">
    <h2>EUMETSAT Conference 2026</h2>
    <p><font color="white"><em>From Data to Insights: working with DestinE Data Lake services</em></font></p>
</div>

<div class="alert alert-block alert-danger">
Extremes DT is only available for the preceeding 15 days. If you are running this notebook after September 30th, you will need to update the context to a newer storm event. This can be done by adapting the according bounding box and time slices, however, the MTG cloud mask product will not correspond with your new data selection unless you replace the content of the <b>../supporting_data/mtg_cloud_mask_downloads</b> directory. These products can be downloaded and prepared using the <b>../supporting_data/prepare_mtg_cloud_background.py</b> script.
</div>

<div class="alert alert-block alert-warning">
<b> Prerequisites: </b><ul>
   <li>To search and access DEDL data a <a href="https://platform.destine.eu/"> DestinE user account</a> is needed</li>
   <li>To search and access DT data an <a href="https://platform.destine.eu/support-pages/access-policy/"> upgraded access</a> is needed.</li></ul>
<b> References: </b><ul>
    <li><a href="https://destine-data-lake-docs.data.destination-earth.eu/en/latest/dedl-discovery-and-data-access/Use-of-Harmonized-Data-Access/Use-of-Harmonized-Data-Access.html">DestinE Data Lake (DEDL) Harmonized Data Access (HDA) documentation</a> </li>
    <li> <a href="https://destine.ecmwf.int/weather-induced-extremes-digital-twin-1/">Weather-Induced Extremes Digital Twin (Extremes DT)</a></li>
    <li> <a href="https://confluence.ecmwf.int/display/DDCZ/Extremes+DT+data+catalogue">Extreme DT data catalogue</a></li></ul>


# Harmonised data access to Extreme Digital Twin output

<div style="margin: 6px 0;">
  <a href="https://jupyter.central.data.destination-earth.eu/user-redirect/lab/tree/DestinE-DataLake-Lab/HDA/DestinE_Digital_Twins/DEDL-HDA-DT_EXTREMES_MTG.ipynb" target="_blank" style="text-decoration: none;">
    <span class="launch">🚀 Launch in JupyterHub</span>
  </a>
</div>

### Data Used

| Dataset | HDA collection ID | HDA collection description |
|:--------------------:|:-----------------------:|:-------------:|
| ECMWF Extremes DT – Convective rain rate | EO.ECMWF.DAT.DT_EXTREMES | <a href="https://data.destination-earth.eu/data-portfolio/EO.ECMWF.DAT.DT_EXTREMES" target="_blank">Description</a> |

This notebook uses the **convective rain rate (`crr`)** parameter (ECMWF parameter ID `228218`) from the Weather-Induced Extremes Digital Twin.

### Learning outcomes

After running this notebook, you will be able to:

* Set up and authenticate access to the DestinE Harmonised Data Access (HDA).
* Discover available parameters and forecast dimensions using the HDA queryables endpoint.
* Construct an HDA request for a specific forecast, parameter and spatial area.
* Use **Polytope spatial extraction** to request only the data required for a defined region.
* Understand how HDA can return a compact **CoverageJSON** representation rather than a complete GRIB field.
* Recognise the difference between Polytope spatial extraction and the alternative HDA workflow for retrieving GRIB data.
* Convert the returned CoverageJSON data into an `xarray` dataset for analysis.
* Visualise a forecast spatially and interactively using a time slider.

### Outline

This notebook introduces the **Harmonised Data Access (HDA)** service of the DestinE Data Lake using the ECMWF **Weather-Induced Extremes Digital Twin**. We will use convective rain rate (`crr`) as an example and request a limited geographic area over central Italy.

The workflow demonstrates an important capability of HDA: data do not always need to be retrieved as a complete global GRIB product. By using **Polytope** to perform the spatial extraction as part of the request, HDA can return only the data required for the area of interest. For a small spatial subset such as the one used here, the resulting CoverageJSON response is enormously smaller than the corresponding GRIB request, making it much more practical for interactive analysis and visualisation.

The notebook also highlights that **GRIB retrieval remains possible through HDA**, but follows a slightly different ordering and retrieval methodology. This provides a useful distinction between obtaining a conventional data product and requesting a compact, spatially extracted representation for a specific application.

The workflow follows these steps:

1. Import the required Python dependencies.
2. Authenticate with HDA and configure the REST endpoints.
3. Inspect the queryables available for the Extremes DT collection.
4. Select the forecast date, parameter, forecast sequence and area of interest.
5. Search HDA and inspect the available retrieval information.
6. Submit a spatial extraction request using Polytope.
7. Poll the asynchronous order until the requested data are ready.
8. Download and inspect the resulting CoverageJSON response.
9. Convert the spatial and temporal data into an `xarray` dataset.
10. Visualise the forecast interactively using a Folium time slider.

<div class="alert alert-info" role="alert">

## Importing dependencies

</div>

We begin by importing the libraries used throughout the notebook. The workflow uses Python packages for authentication and HTTP requests, together with `xarray` for structuring the returned data and `Folium` for interactive visualisation.


In [ ]:
import destinelab as deauth                     # DestinE Data Lake authentication and access
import json                                     # Read and write JSON data
import time                                     # Pause between polling requests
import os                                       # Interact with the operating system
import shutil                                   # Remove previous download files
import requests                                 # Make HTTP requests to web services and APIs
from getpass import getpass                     # Securely prompt for passwords
from datetime import datetime, timedelta        # Work with dates and time intervals
import re                                       # Extract timestamps from MTG filenames
from pathlib import Path                        # Construct and manage system file paths
from IPython.display import JSON                # Display JSON data in Jupyter notebooks
import numpy as np                              # Work with numerical arrays
import xarray as xr                             # Work with multidimensional data
import folium                                   # Create interactive maps
from folium.plugins import TimestampedGeoJson   # Animate time-varying geographic data
import matplotlib.pyplot as plt                 # Access colour maps
from matplotlib.colors import Normalize         # Scale values to a colour range
from satpy import Scene                         # Read and process MTG satellite data
from pyresample.geometry import AreaDefinition  # Define the target area for resampling
import warnings                                 # Control warning messages

warnings.filterwarnings("ignore")

Before starting the workflow, we remove any files from a previous run and create a clean download directory.


In [ ]:
# Start with a clean download directory
shutil.rmtree("hda_downloads", ignore_errors=True)
Path("hda_downloads").mkdir()

In [ ]:
# Optional MTG FCI Cloud Mask background
USE_MTG_BACKGROUND = True
MTG_BACKGROUND_DIR = Path("../Supporting_data/mtg_cloud_mask_downloads")
MTG_BACKGROUND_STRIDE = 20  # Reduce the resampled grid for efficient interactive mapping

<div class="alert alert-info" role="alert">

## Authenticating the HDA

</div>

HDA requires authentication for data access. For this training notebook we store the credentials locally in a file under the user's home directory so that they do not need to be entered every time the notebook is run.

If the credentials file does not yet exist, the notebook prompts for the DESP username and passwords and creates it. The password prompts use `getpass`, so the credentials are not displayed in the notebook.

**Security note:** this file contains sensitive credentials and should never be committed to a source-control repository or shared with other users.

In [ ]:
credentials_file = Path(Path.home() / '.dedl' / 'credentials')

if os.path.exists(credentials_file):
    with open(credentials_file, "r") as f:
        config = json.load(f)
    DESP_USERNAME = config["username"]
    DESP_PASSWORD = config["desp_password"]
else:
    # creating authentication file
    DESP_USERNAME = input("Please input your DESP username or email: ")
    DESP_PASSWORD = getpass("Please input your DESP password: ")
    OIDC_PASSWORD = getpass("Please input your OIDC password (if known): ")

    config = {"username": DESP_USERNAME,
              "desp_password": DESP_PASSWORD,
              "oidc_password": OIDC_PASSWORD}
    try:
        os.makedirs(os.path.dirname(credentials_file), exist_ok=True)        
        with open(credentials_file, "w") as f:
            json.dump(config, f, indent=4)
    except:
        pass

The credentials themselves are not sent directly with every HDA request. Instead, they are exchanged for an **access token** using the DestinE authentication service.

The token is then placed in an HTTP `Authorization` header. This header is reused for the authenticated HDA requests that follow.

In [ ]:
auth = deauth.AuthHandler(DESP_USERNAME, DESP_PASSWORD)
access_token = auth.get_token()

if access_token is not None:
    print("DEDL/DESP Access Token Obtained Successfully")
else:
    print("Failed to Obtain DEDL/DESP Access Token")

auth_headers = {"Authorization": f"Bearer {access_token}"}

Before continuing, we check that the account has access to Digital Twin data.


In [ ]:
auth.is_DTaccess_allowed(access_token)

<div class="alert alert-info" role="alert">

## Setting HDA endpoints

</div>

HDA exposes a REST API whose catalogue and discovery interfaces follow the **SpatioTemporal Asset Catalog (STAC)** specification. We define the STAC endpoint and the collection ID for the Weather-Induced Extremes Digital Twin.


In [ ]:
HDA_API_URL = "https://hda.data.destination-earth.eu"
STAC_API_URL = f"{HDA_API_URL}/stac/v2"
COLLECTION_ID = "EO.ECMWF.DAT.DT_EXTREMES"

The collection-specific **queryables** endpoint exposes the filters available for the Extremes DT data. We use it to determine the valid values for the parameter and its associated forecast dimensions.


In [ ]:
QUERYABLES_URL = f"{STAC_API_URL}/collections/{COLLECTION_ID}/queryables"

response = requests.get(QUERYABLES_URL, headers=auth_headers)
response.raise_for_status()

print(f"HDA queryables endpoint: {QUERYABLES_URL}")
JSON(response.json())

<div class="alert alert-info" role="alert">

## HDA, Polytope and spatial extraction

</div>

HDA provides a common access layer to the Digital Twin data, while the **Polytope** service is used to extract the requested spatial subset. In this example, we use a Polytope `boundingbox` feature to ask HDA for only the geographic area of interest.

This is particularly useful for large Digital Twin fields: instead of downloading a complete global GRIB field and subsetting it locally, the spatial extraction is performed as part of the HDA request. For this example, the result is returned as **CoverageJSON**, which is enormously smaller than the corresponding global GRIB request and is well suited to interactive visualisation.

A GRIB extraction is also possible through HDA. The ordering workflow is similar, but the request uses the GRIB-oriented ordering methodology rather than the Polytope spatial-extraction approach demonstrated here. The two approaches therefore provide different representations of the same underlying forecast data, depending on whether a compact spatial subset or a conventional GRIB product is required.


### Selecting the parameter

The queryables response contains the available filters for the collection. We first refine the queryables response using the ECMWF parameter ID for **convective rain rate (`crr`)**, `228218`. You can find reference to the available parameters in the <a href="https://confluence.ecmwf.int/spaces/DDCZ/pages/505395619/Extremes+DT+data+catalogue" target="_blank">Extremes DT data catalogue</a>.


In [ ]:
PARAMETER_ID = "228218"

response = requests.get(
    QUERYABLES_URL,
    params={"ecmwf:param": PARAMETER_ID},
    headers=auth_headers
)
response.raise_for_status()

JSON(response.json())

The parameter-specific queryables tell us which forecast dimensions apply to this parameter. In particular, the available `levtype`, `stream` and `time` values can be used to construct the search request.


### Selecting dates

Extremes DT forecasts are available only within a recent time window. We therefore use the dates exposed by the queryables response and select the most recent available date within the last 15 days.

In [ ]:
current_date = datetime.utcnow()
date_15_days_ago = current_date - timedelta(days=15)

available_dates = response.json()["properties"]["ecmwf:date"]["enum"]
dates = [datetime.strptime(d, "%Y%m%d") for d in available_dates]

dates_in_range = [
    d for d in dates
    if date_15_days_ago <= d <= current_date
]

lastdate = dates_in_range[-1].date().isoformat()

print(f"Last available Extremes DT run: {lastdate}")

We use the most recent available forecast date identified above as the target date for the example.


In [ ]:
target_date = datetime(2026,9,18).date().isoformat()
print(f"Using forecast initialised on: {target_date}")

### Selecting the forecast sequence and target area

We request a **3-hourly forecast sequence covering 48 hours** and define a geographic bounding box for the area we want to download.

The bounding box is expressed as **West, South, East, North**. It is then converted to the Polytope `boundingbox` feature used by HDA.

Digital Twin searches are asynchronous. Rather than returning the data immediately, the search returns a single **orderable STAC Item** containing the endpoint and request body required to order the selected data.

In [ ]:
# Forecast sequence and target area
steps = [str(step) for step in range(0, 49, 3)]

bbox = [12.5, 42.6, 14.5, 44.6]  # West, South, East, North
west, south, east, north = bbox

feature = {
    "type": "boundingbox",
    "points": [[north, west], [south, east]]
}

We can now submit the STAC search. The search identifies the orderable product matching the requested parameter, forecast date, forecast dimensions and time steps.

The spatial `boundingbox` feature is **not** part of this STAC search. We add it to the `retrieve` request returned by the search, so that the spatial extraction is applied when the data are ordered.


We construct the HDA query from the parameter-specific forecast dimensions and the selected forecast steps.


In [ ]:
query_params = {
    "ecmwf:type": "fc",
    "ecmwf:levtype": response.json()["properties"]["ecmwf:levtype"]["const"],
    "ecmwf:param": [PARAMETER_ID],
    "ecmwf:stream": response.json()["properties"]["ecmwf:stream"]["const"],
    "ecmwf:step": steps,
    "ecmwf:time": [response.json()["properties"]["ecmwf:time"]["items"]["const"]]
}

hda_filters = {key: {"eq": value} for key, value in query_params.items()}
JSON(hda_filters)

The search request is submitted to the HDA STAC `/search` endpoint using the selected collection, forecast date and query filters.


In [ ]:
session = requests.Session()

search_response = session.post(
    f"{STAC_API_URL}/search",
    headers=auth_headers,
    json={
        "collections": [COLLECTION_ID],
        "datetime": f"{target_date}T00:00Z",
        "query": hda_filters
    }
)
search_response.raise_for_status()

We inspect the search response and select the matching STAC Item that can be ordered.


In [ ]:
features = search_response.json().get("features", [])

if not features:
    raise ValueError("No matching Extremes DT product was found.")

product = features[0]
JSON(product)

The returned STAC Item contains the information needed to order the data. Its `retrieve` link provides the HDA order endpoint and request body. We add the Polytope `boundingbox` feature to this order so that the extraction is spatially limited before the data are returned.


In [ ]:
link = next(link for link in product["links"] if link.get("rel") == "retrieve")

href = link["href"]
body = link["body"]
body["feature"] = feature

print(f"Order endpoint: {href}")
JSON(body)

<div class="alert alert-info" role="alert">

## Ordering and downloading the data

</div>

ECMWF Digital Twin data in the DestinE Data Lake uses an **asynchronous ordering workflow**. HDA first accepts the request and creates an order; the notebook then checks the order status until the data are ready.

Here, the order contains the Polytope `boundingbox` feature. This means the spatial extraction happens as part of the HDA request rather than after downloading a global field. The response is therefore a compact spatial dataset rather than the much larger global GRIB representation.

### Submit the order

In [ ]:
response = session.post(href, json=body, headers=auth_headers)
response.raise_for_status()

ordered_item = response.json()
product_id = ordered_item["id"]

print(f"Product ordered: {product_id}")
print(f"Order status: {ordered_item['properties'].get('order:status', 'unknown')}")

### Poll until the product is ready

The order initially has a status such as `orderable` or `running`. We repeatedly request the corresponding STAC Item until its `order:status` becomes `succeeded`.

In [ ]:
self_url = f"{STAC_API_URL}/collections/{COLLECTION_ID}/items/{product_id}"

for _ in range(150):
    item = session.get(self_url, headers=auth_headers).json()
    status = item["properties"].get("order:status")
    print(f"Order status: {status}")

    if status == "succeeded":
        download_url = item["assets"]["downloadLink"]["href"]
        break

    time.sleep(2)
else:
    raise TimeoutError("Product was not ready after 5 minutes.")

### Download the product

Once the order has succeeded, the `downloadLink` asset provides the requested spatial subset as **CoverageJSON**. This is the compact representation produced by the Polytope spatial extraction.

In [ ]:
download_dir = Path("hda_downloads")
output_path = download_dir / "extremes_dt_subset.json"

with session.get(download_url, headers=auth_headers) as response:
    response.raise_for_status()
    output_path.write_bytes(response.content)

print(f"Downloaded: {output_path}")
print(f"Size: {output_path.stat().st_size / 1024 / 1024:.2f} MB")

<div class="alert alert-info" role="alert">

## Plotting our data

</div>

The spatial subset is returned as **CoverageJSON** rather than a GRIB file. Each coverage represents one forecast step and contains the latitude/longitude points and the corresponding `crr` values. We convert these point data into a compact `xarray.Dataset` for plotting.

We load the downloaded CoverageJSON file and access the individual forecast-step coverages.


In [ ]:
with open(output_path) as f:
    coverage = json.load(f)

coverages = coverage["coverages"]

Each coverage contains one forecast step. The `composite` coordinates provide the latitude and longitude of the sampled points, while the `crr` range contains the corresponding convective rain-rate values.

In [ ]:
steps = []
values = []
latitudes = []
longitudes = []

for item in coverages:
    steps.append(int(item["mars:metadata"]["step"]))

    points = np.asarray(
        item["domain"]["axes"]["composite"]["values"],
        dtype=float
    )

    values.append(np.asarray(item["ranges"]["crr"]["values"], dtype=float))
    latitudes.append(points[:, 0])
    longitudes.append(points[:, 1])

We combine the coverage data into an `xarray.Dataset`, keeping forecast step, valid time and geographic coordinates aligned with the `crr` values.


In [ ]:
valid_times = [
    datetime.fromisoformat(target_date) + timedelta(hours=step)
    for step in steps
]

ds = xr.Dataset(
    {"crr": (("step", "point"), np.asarray(values))},
    coords={
        "step": ("step", np.asarray(steps)),
        "valid_time": ("step", np.asarray(valid_times, dtype="datetime64[ns]")),
        "latitude": (("step", "point"), np.asarray(latitudes)),
        "longitude": (("step", "point"), np.asarray(longitudes)),
    },
)

The spatial subset has already been applied by HDA through the Polytope `boundingbox` feature. The downloaded data therefore contains only the points needed for the target area — no global field is downloaded or cropped locally.

The Polytope bounding box is represented as a collection of geographic points rather than a regular raster grid. That reflects the underlying point-based spatial representation, so we visualise the `crr` values as coloured points rather than trying to force the data into a raster.

We now turn the forecast sequence into an interactive map. A single colour scale is applied to all forecast steps so that changes in rainfall intensity remain comparable through time.

The Folium `TimestampedGeoJson` plugin provides the time slider and playback controls. Zero-rainfall points are omitted from the display, while each non-zero point is coloured according to its `crr` value.

### Setting up the interactive map

We create a Folium map centred on the area of interest and define the colour scale for convective rain rate (crr).

The same scale is used for every forecast step, so changes in rainfall intensity can be compared consistently through time.

In [ ]:
vmax = float(ds.crr.quantile(0.995))
cmap = plt.get_cmap("Blues")
norm = Normalize(vmin=0, vmax=vmax)
forecast_date = datetime.fromisoformat(target_date)

map1 = folium.Map(
    location=[(south + north) / 2, (west + east) / 2],
    zoom_start=6,
    tiles="OpenStreetMap"
)

folium.Rectangle(
    bounds=[[south, west], [north, east]],
    color="red",
    weight=2,
    fill=False,
    tooltip="HDA spatial subset"
).add_to(map1);

### Adding the MTG Cloud Mask

We use the corresponding MTG FCI Cloud Mask observation as a background layer for each forecast time.

The satellite data are resampled to the same area as the forecast and displayed using a simple discrete mask: clear pixels are transparent, while cloud classes are shown in dark and light red.

This provides observational context for the rainfall forecast without obscuring the forecast itself.

In [ ]:
features = []

if USE_MTG_BACKGROUND:
    mtg_files = sorted(MTG_BACKGROUND_DIR.rglob("*.nc"))
    mtg_by_time = {}

    for path in mtg_files:
        match = re.search(r"_OPE_(\d{14})_", path.name)
        if match:
            mtg_by_time[datetime.strptime(match.group(1), "%Y%m%d%H%M%S")] = path

    target_area = AreaDefinition(
        "roi", "ROI", "roi", {"proj": "longlat", "datum": "WGS84"},
        1000, 1000, tuple(bbox)
    )
    lons, lats = target_area.get_lonlats()
    dlat = float(np.nanmedian(np.abs(np.diff(lats[:, 0]))))
    dlon = float(np.nanmedian(np.abs(np.diff(lons[0, :]))))
    colours = {1: "#8b0000", 2: "#ff9999"}

    for step in ds.step.values:
        valid_time = forecast_date + timedelta(hours=int(step))
        nc_path = mtg_by_time.get(valid_time)
        if nc_path is None:
            continue

        scene = Scene(filenames=[str(nc_path)], reader="fci_l2_nc")
        scene.load(["cloud_state"])
        cloud = scene.resample(target_area, resampler="nearest")["cloud_state"].values
        timestamp = valid_time.isoformat()
        half_lat, half_lon = dlat * MTG_BACKGROUND_STRIDE / 2, dlon * MTG_BACKGROUND_STRIDE / 2

        for iy in range(0, cloud.shape[0], MTG_BACKGROUND_STRIDE):
            for ix in range(0, cloud.shape[1], MTG_BACKGROUND_STRIDE):
                colour = colours.get(int(cloud[iy, ix]))
                if colour is None:
                    continue
                x, y = float(lons[iy, ix]), float(lats[iy, ix])
                features.append({
                    "type": "Feature",
                    "geometry": {"type": "Polygon", "coordinates": [[
                        [x-half_lon, y-half_lat], [x+half_lon, y-half_lat],
                        [x+half_lon, y+half_lat], [x-half_lon, y+half_lat],
                        [x-half_lon, y-half_lat]
                    ]]},
                    "properties": {
                        "times": [timestamp],
                        "style": {"color": colour, "weight": 0,
                                  "fillColor": colour, "fillOpacity": 0.9}
                    }
                })

### Adding the forecast rainfall

The Extremes DT crr values are added as timestamped points on top of the satellite cloud mask.

Only non-zero rainfall is shown, with darker shades of blue representing higher rain rates. The two layers therefore provide a simple comparison between observed cloud conditions and forecast precipitation.

In [ ]:
for frame, step in enumerate(ds.step.values):
    rain = ds.crr.isel(step=frame).values
    lon = ds.longitude.isel(step=frame).values
    lat = ds.latitude.isel(step=frame).values
    timestamp = (forecast_date + timedelta(hours=int(step))).isoformat()

    for x, y, value in zip(lon[rain > 0], lat[rain > 0], rain[rain > 0]):
        rgba = cmap(norm(float(value)))
        colour = "#{:02x}{:02x}{:02x}".format(*(int(c * 255) for c in rgba[:3]))
        features.append({
            "type": "Feature",
            "geometry": {"type": "Point", "coordinates": [float(x), float(y)]},
            "properties": {
                "times": [timestamp],
                "icon": "circle",
                "iconstyle": {"fillColor": colour, "fillOpacity": 0.75,
                              "stroke": False, "radius": 4}
            }
        })

### Animate the forecast

Finally, we combine the two layers in a single time-enabled GeoJSON layer. The time slider allows us to step through the forecast and compare the MTG cloud mask and Extremes DT precipitation at each 3-hourly forecast time.

In [ ]:
TimestampedGeoJson(
    {
        "type": "FeatureCollection",
        "features": features
    },
    period="PT3H",
    duration="PT3H",
    add_last_point=False,
    auto_play=False,
    loop=False,
    max_speed=1,
    loop_button=True,
    date_options="YYYY-MM-DD HH:mm",
    time_slider_drag_update=True
).add_to(map1)

map1

<div class="alert alert-info" role="alert">

## Conclusion

</div>

This notebook has taken us through the **ECMWF Extremes DT HDA workflow**:

**authenticate → inspect queryables → select a parameter, date and area → search → order → poll → download → visualise**

The example uses HDA together with a Polytope `boundingbox` feature to request a **3-hourly, 48-hour forecast sequence for a small geographic area**. Because the spatial extraction is performed before delivery, the result is a compact **CoverageJSON** dataset rather than a large global GRIB file. This makes the example particularly suitable for interactive, browser-based exploration.

The underlying data can also be requested as **GRIB** through HDA using the GRIB-oriented ordering methodology. That approach is useful when the conventional GRIB representation or a larger/global field is required; the Polytope approach demonstrated here is useful when a compact spatial subset is sufficient.

For further information, see:
* <a href="https://destine-data-lake-docs.data.destination-earth.eu/en/latest/dedl-discovery-and-data-access/Use-of-Harmonized-Data-Access/Use-of-Harmonized-Data-Access.html" target="_blank">DestinE Data Lake HDA documentation</a>
* <a href="https://confluence.ecmwf.int/spaces/DDCZ/pages/505395619/Extremes+DT+data+catalogue" target="_blank">Extremes DT data catalogue</a>
* <a href="https://destine.ecmwf.int/weather-induced-extremes-digital-twin-1/" target="_blank">Weather-Induced Extremes Digital Twin</a>


<img src='https://github.com/destination-earth/DestinE-DataLake-Lab/blob/main/img/DestinE-banner.jpg?raw=true' align='right' width='100%'/>
